# 03 — Comparative evaluation: base vs few-shot vs fine-tuned (× constrained)

Runs **3 settings × 2 test sets** and prints a single results table. Constrained decoding is added as a fourth row on top of the fine-tuned model.

Settings:
1. Base model, zero-shot
2. Base model, 3-shot
3. Fine-tuned (QLoRA adapter)
4. Fine-tuned + constrained decoding (outlines)

Test sets:
- `test.parquet` — 20 MACCROBAT docs held out during training
- `test_unseen_docs.parquet` — 20 further MACCROBAT docs never touched

Success criterion: fine-tuned wins on every metric in both test sets; cross-test gap is small (good generalisation).

In [ ]:
import os, sys

def _find_project_root() -> str:
    candidates = [os.path.abspath("..")]
    candidates += [
        "/content/fineTune LLM",
        "/content/fineTune_LLM",
        "/content/finetune-llm",
        "/content/drive/MyDrive/fineTune LLM",
        "/content/drive/MyDrive/fineTune_LLM",
    ]
    if os.path.isdir("/content"):
        for entry in os.listdir("/content"):
            p = os.path.join("/content", entry)
            if os.path.isfile(os.path.join(p, "schema.py")):
                candidates.append(p)
    for c in candidates:
        if os.path.isfile(os.path.join(c, "schema.py")):
            return c
    return ""

PROJECT_ROOT = _find_project_root()
if not PROJECT_ROOT:
    raise FileNotFoundError(
        "Project files not found. On Colab, either:\n"
        "  (A) Clone your repo:\n"
        "      !git clone https://github.com/<you>/<repo>.git '/content/fineTune LLM'\n"
        "  (B) Upload via the file panel (left sidebar) to /content/fineTune LLM/\n"
        "  (C) Mount Drive and copy from MyDrive.\n"
        "Then re-run this cell."
    )
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working in:", os.getcwd())


with open("artifacts/gate_decisions.json") as f:
    gate = json.load(f)
base_model_name = gate["target_model"]
slug = base_model_name.split("/")[-1].lower().replace(".", "")
adapter_dir = f"artifacts/{slug}-meds-qlora"
print("Base:", base_model_name)
print("Adapter:", adapter_dir)

In [ ]:
import pandas as pd
from src.eval import evaluate

test_sets = {
    "standard": pd.read_parquet("data/test.parquet"),
    "unseen_docs": pd.read_parquet("data/test_unseen_docs.parquet"),
}
for name, df in test_sets.items():
    print(f"{name}: {len(df)} examples, {df['doc_id'].nunique()} docs")

In [ ]:
# ---- Base model loading (used for zero-shot and 3-shot) ----
from unsloth import FastLanguageModel
base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=1024,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

In [ ]:
from src.generate import generate as gen_vanilla

rows = []
for test_name, df in test_sets.items():
    inputs = df["input"].tolist()
    golds = df["target"].tolist()
    for n_shots, label in [(0, "base_zeroshot"), (3, "base_3shot")]:
        preds = gen_vanilla(base_model, base_tok, inputs, n_shots=n_shots)
        res = evaluate(preds, golds)
        row = {"setting": label, "test_set": test_name, **res.to_row()}
        rows.append(row)
        print(label, test_name, res.to_row())

In [ ]:
# free base-model VRAM before loading fine-tuned version
import gc, torch
del base_model
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- Fine-tuned model (load adapter on top of 4-bit base) ----
from unsloth import FastLanguageModel
ft_model, ft_tok = FastLanguageModel.from_pretrained(
    model_name=adapter_dir,
    max_seq_length=1024,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)

In [ ]:
for test_name, df in test_sets.items():
    inputs = df["input"].tolist()
    golds = df["target"].tolist()
    preds = gen_vanilla(ft_model, ft_tok, inputs, n_shots=0)
    res = evaluate(preds, golds)
    row = {"setting": "fine_tuned", "test_set": test_name, **res.to_row()}
    rows.append(row)
    print("fine_tuned", test_name, res.to_row())

In [ ]:
# ---- Fine-tuned + constrained decoding (outlines) ----
from src.generate_constrained import generate_outlines, generate_lmfe
for test_name, df in test_sets.items():
    inputs = df["input"].tolist()
    golds = df["target"].tolist()
    try:
        preds = generate_outlines(ft_model, ft_tok, inputs)
        backend = "outlines"
    except Exception as e:
        print(f"outlines failed ({e}); falling back to lm-format-enforcer")
        preds = generate_lmfe(ft_model, ft_tok, inputs)
        backend = "lmfe"
    res = evaluate(preds, golds)
    row = {"setting": f"fine_tuned_constrained_{backend}", "test_set": test_name, **res.to_row()}
    rows.append(row)
    print(row["setting"], test_name, res.to_row())

In [ ]:
# ---- Final table ----
import pandas as pd
results = pd.DataFrame(rows)
results_pivot = results.pivot(index="setting", columns="test_set")
print(results_pivot.to_string())
results.to_csv("artifacts/eval_results.csv", index=False)
print("\nSaved artifacts/eval_results.csv")

## Interactive smoke test

Quick confidence check on a hand-written prompt. The fine-tuned model should emit two medications: metoprolol (all fields filled) and lisinopril (duration null).

In [ ]:
test_sentence = "Start patient on metoprolol tartrate 25 mg PO BID for 2 weeks, then add lisinopril 10 mg daily."
preds = gen_vanilla(ft_model, ft_tok, [test_sentence])
print(preds[0])